# Supplementary 2 Parameters recovery and posterior predictive checks

## Parameters recovery
To ensure that our model is capable of accurately estimate the parameters from the data, we conduct parameters recovery tests in which we simulate data based on ground truth parameters, which we then compare to what the model recovers. We simulate data with different parameters magnitudes to showcase the sensitivity of our model. 

### Preparing the data
To make sure that the parameters recovery analysis reflects the model capacity to retrieve parameters in our own data, we simulate data based on the exact experimental design we've used to collect the data. Accordingly, we first prepare the data in the exact same way we did in our main analysis.

In [1]:
# General utilities:
import os 
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings

# Stats
import pymc as pm
import arviz as az
import bambi as bmb
from scipy.special import expit

# Custom packages:
from stabst.MarkovDecisionProcess import MDP
from stabst.TaskConfig import LimitedEnergyTask
from stabst.utils import avg_reduce_mdp, abstract2ground_value

from utilities.models import preference_model

# Download the data if needed:
if not os.path.exists('./data/raw_data/all_participants_data.csv'):
    if not os.path.exists('./data/raw_data'):
        os.makedirs('./data/raw_data')
    url = 'https://raw.githubusercontent.com/fmott/context_dependent_planning/4d239b721749adabb8fe8f1d8ac2d1ecdeba17cf/data/behaviour/data_all_participants_20220215120148.csv'
    os.system(f'wget {url} -O ./data/raw_data/all_participants_data.csv')
if not os.path.exists('./data/raw_data/all_participants_age_gender.csv'):
    if not os.path.exists('./data/raw_data'):
        os.makedirs('./data/raw_data')
    url = 'https://raw.githubusercontent.com/fmott/context_dependent_planning/4d239b721749adabb8fe8f1d8ac2d1ecdeba17cf/data/behaviour/age_gender.csv'
    os.system(f'wget {url} -O ./data/raw_data/all_participants_age_gender.csv')
# Load the data:
beh_data = pd.read_csv('./data/raw_data/all_participants_data.csv')
demographic_data = pd.read_csv('./data/raw_data/all_participants_age_gender.csv', sep=";")

# Extract demographic information:
n_participants = demographic_data.shape[0]
n_female = demographic_data['gender (m = 1, f = 2)'].value_counts()[2]
mean_age = demographic_data['age'].mean()
std_age = demographic_data['age'].std()

# ===================================================================
# Data preprocessing:
# Remove nans:
beh_data = beh_data.dropna()
# Remove timeout:
beh_data = beh_data[beh_data["timeout"] == 0]
# Flip responses: 1 = accept:
beh_data["response"] = (beh_data["response"] == 0).astype(int)
# Make trial 1 based
beh_data["trial"] = beh_data["trial"] + 1
# Generate future cost based on the transitions:
transitions_costs = {
    0: [1, 1],
    1: [2, 1],
    2: [1, 2],
    3: [2, 2]
}
beh_data["fc"] = [transitions_costs[row["transition"]][1] for _, row in beh_data.iterrows()]

# Categorical costs regressor
beh_data['is_lc'] = (beh_data['energy_cost'] == 1).astype(int).to_numpy()
beh_data['is_hc'] = (beh_data['energy_cost'] == 2).astype(int).to_numpy()
# Categorical future costs regressor
beh_data['is_lfc'] = (beh_data['fc'] == 1).astype(int).to_numpy()
beh_data['is_hfc'] = (beh_data['fc'] == 2).astype(int).to_numpy()

# Categorical energy regressor:
beh_data['e_is_0'] = (beh_data['energy'] == 0).to_numpy()
beh_data['e_is_1'] = (beh_data['energy'] == 1).to_numpy()
beh_data['e_is_2'] = (beh_data['energy'] == 2).to_numpy()
beh_data['e_is_3'] = (beh_data['energy'] == 3).to_numpy()
beh_data['e_is_4'] = (beh_data['energy'] == 4).to_numpy()
beh_data['e_is_5'] = (beh_data['energy'] == 5).to_numpy()
beh_data['e_is_6'] = (beh_data['energy'] == 6).to_numpy()

# Random effects
subj_idx_raw, subj_labels = pd.factorize(beh_data["vpn"])

# ===================================================================
# Compute decision values:
# Create the task and its parameters (transition probability, reward...):
task = LimitedEnergyTask(O=[1, 2, 3, 4], p_offer=[1/4] * 4)
task.build()

# Create full MDP and compute solution for later reference:
gamma = 1
task_mdp = MDP(task.states, task.tp, task.r, gamma, s2i=task.s2i)
V_full, Q_full = task_mdp.backward_induction()

# Add decision values to the data frame:
dv = Q_full[:, 1] - Q_full[:, 0]
# Loop through each trial to set DV:
dv_trials = []
for trial_i, trial in beh_data.iterrows():
    e, o, cc, t = trial.energy, trial.reward, trial.energy_cost, trial.trial
    fc = transitions_costs[trial.transition][1]
    dv_trials.append(dv[task.s2i[(e, o, cc, fc, t)]])
beh_data['dv'] = dv_trials

### Simulation
Next, we simulate data where the effects associated with each parameter varies from 0 (absence of an effect) to 2, in increments of 0.2, while the variance is fixed to be equal to 1. 

In [2]:
simulated_data = {}
# Extract the data:
pref_cols = [
    'is_1', 'is_2', 'is_3', 'is_4', 
    'is_lc', 'is_hc', 
    'is_lfc', 'is_hfc', 
    'e_is_0', 'e_is_1', 'e_is_2', 'e_is_3', 'e_is_4', 'e_is_5', 'e_is_6'
]

# Loop through each effect sizes:

for fsize in np.linspace(0, 2, 11):
    fsize_data = []
    for sub in beh_data["vpn"].unique():
        # Extract the data from this subject:
        subject_data = beh_data[beh_data["vpn"] == sub]
        # Extract the preferences regressors:
        prefs_reg = subject_data[pref_cols].to_numpy()

        # Randomly pick parameters:
        beta_pref = np.random.normal(fsize, scale=2, size=len(pref_cols))
        beta_plan = np.random.normal(fsize, scale=2, size=1)
        beta_interaction = np.random.normal(fsize, scale=2, size=1)

        # Preference component:
        pref_score = (beta_pref * prefs_reg).sum(axis=-1).astype(float)
        # Planning component:
        planning = beta_plan * subject_data["dv"]
        # Interaction component:
        entropy = -expit(pref_score) * np.log(expit(pref_score)) - (1-expit(pref_score)) * np.log(1 - expit(pref_score))
        interaction = beta_interaction * (subject_data["dv"] * entropy)

        # Compute responses:
        responses = [np.random.binomial(1, expit(eta)) for eta in planning + pref_score + interaction]
        # Combine in a dataframe:
        df = pd.DataFrame({
                "dv": subject_data["dv"],
                **{col: subject_data[col] for col in pref_cols},
                "subject": [sub] * len(subject_data),
                "response": responses,
                }
            )
        fsize_data.append(df)

    # Concatenate the data across subjects:
    simulated_data[fsize] = pd.concat(fsize_data)

### Fitting the model and comparing estimated to known parameters
If all goes well, we would expect the estimated parameters to be close to the known ground truth parameters, provided that the effects are large enoug that they can be disentangled from the noise. To test that this is the case, we will fit the preference model to our simulated data for each parameters amplitude separately. We would expect the population level for each participant ot be close to the effect amplitude we have set in the simulation:

In [3]:
traces = {}
# Looping through each effect sizes:
for fsize in simulated_data.keys():
    # Extract the data for this effect size:
    fsize_data = simulated_data[fsize]
    # Fit the model:
    idata = preference_model(
        y=fsize_data["response"].values,
        decision_values=fsize_data["dv"].values,
        pref_regressors=fsize_data[pref_cols],
        subject_index=subj_idx_raw,
        subject_labels=subj_labels
    )
    # Store traces:
    traces[fsize] = idata

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_intercept, sigma_intercept, beta_planning, sigma_planning, beta_pref, sigma_pref, beta_interaction, sigma_interaction, z_intercept, z_planning, z_biases, z_interaction]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 2072 seconds.
There were 2 divergences after tuning. Increase `target_accept` or reparameterize.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_intercept, sigma_intercept, beta_planning, sigma_planning, beta_pref, sigma_pref, beta_interaction, sigma_interaction, z_intercept, z_planning, z_biases, z_interaction]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1939 seconds.
There were 13 divergences after tuning. Increase `target_accept` or reparameterize.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_intercept, sigma_intercept, beta_planning, sigma_planning, beta_pref, sigma_pref, beta_interaction, sigma_interaction, z_intercept, z_planning, z_biases, z_interaction]


...

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1277 seconds.
There were 3 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_intercept, sigma_intercept, beta_planning, sigma_planning, beta_pref, sigma_pref, beta_interaction, sigma_interaction, z_intercept, z_planning, z_biases, z_interaction]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1234 seconds.
There were 5 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_intercept, sigma_intercept, beta_planning, sigma_planning, beta_pref, sigma_pref, beta_interaction, sigma_interaction, z_intercept, z_planning, z_biases, z_interaction]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1144 seconds.
There were 91 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



## Posterior predictive checks
To ensure that the models that we have fitted to the data were able to extract relevant information, we perform posterior predictive checks. Specifically, we perform out of sample predictions to highlight that the key features present in the data can be replicated by our model. 